In [7]:
import os
import platform
import sqlite3
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
import re


COLUNA_DATA = "FENTREGA"
COLUNA_ORIGEM = "ficheiro_origem"
NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"
REGEX_ANO = re.compile(r"^(\d{4})")



In [8]:
if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00_DB\2026.duckdb"
    PASTA_FICHEIROS = Path(r"C:\Users\LISARR\Desktop\INFORM_27_XLS")
    PASTA_FICHEIROS_OUTPUT = Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\000.Dados_input\inform_27")
elif platform.system() == 'Darwin':
    DB_PATH = "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00.DB/2026.db"
    PASTA_FICHEIROS = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados")
    PASTA_FICHEIROS_OUTPUT = Path("/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados_output")

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ BD: {DB_PATH}")


✓ BD: C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00_DB\2026.duckdb


In [9]:
import csv

# ==========================
# Converter XLS → CSV (Streaming, sem carregar tudo em RAM)
# ==========================
def ler_xls_streaming(caminho):
    """Retorna (cabecalho, generator de linhas)."""
    tree = ET.parse(caminho)
    raiz = tree.getroot()
    worksheets = raiz.findall(".//ss:Worksheet", NS)

    if not worksheets:
        raise ValueError("Nenhuma worksheet encontrada.")

    def ler_linha(row):
        valores = []
        proximo = 1
        for cell in row.findall("ss:Cell", NS):
            indice = cell.get(SS_INDEX)
            indice = int(indice) if indice else proximo
            while len(valores) < indice - 1:
                valores.append(None)
            data = cell.find("ss:Data", NS)
            valores.append(data.text if data is not None else None)
            proximo = indice + 1
        return valores

    # Extrair cabeçalho da 1ª sheet
    primeira_tabela = worksheets[0].find("ss:Table", NS)
    primeira_row = primeira_tabela.findall("ss:Row", NS)[0]
    cabecalho = ler_linha(primeira_row)
    cabecalho = [str(v).strip() if v else f"COLUNA_{i + 1}" for i, v in enumerate(cabecalho)]

    def gerar_linhas():
        for num_sheet, worksheet in enumerate(worksheets):
            tabela = worksheet.find("ss:Table", NS)
            if tabela is None:
                continue
            rows = tabela.findall("ss:Row", NS)
            if not rows:
                continue
            linhas_comeco = 1 if num_sheet == 0 else 0
            for row in rows[linhas_comeco:]:
                valores = ler_linha(row)
                if len(valores) < len(cabecalho):
                    valores += [None] * (len(cabecalho) - len(valores))
                yield valores[:len(cabecalho)]

    return cabecalho, gerar_linhas()


ficheiros = sorted(PASTA_FICHEIROS.rglob("*.xls"))

if not ficheiros:
    print(f"⚠️  Nenhum ficheiro .xls em: {PASTA_FICHEIROS}")
else:
    total_convertidos = 0
    total_linhas = 0
    colunas_numericas = ["INGRESODT", "COSTEDT", "RENTADT", "PALETSDT"]
    PASTA_FICHEIROS_OUTPUT.mkdir(parents=True, exist_ok=True)
    caminho_output = PASTA_FICHEIROS_OUTPUT / "SAL_DAT027.csv"

    with open(caminho_output, "w", newline="", encoding="utf-8", buffering=8192) as f:
        writer = None
        cabecalho_base = None

        for numero, caminho in enumerate(ficheiros, 1):
            try:
                contador = 0
                cabecalho, linhas_gen = ler_xls_streaming(caminho)

                # Escrever o cabeçalho apenas uma vez
                if writer is None:
                    writer = csv.writer(f, delimiter=';')
                    cabecalho_base = cabecalho
                    writer.writerow(cabecalho_base)
                elif cabecalho != cabecalho_base:
                    raise ValueError("Cabeçalho diferente do primeiro ficheiro.")

                # Encontrar índices das colunas
                idx_gestion = cabecalho.index("GESTION") if "GESTION" in cabecalho else None
                idx_numericas = {col: cabecalho.index(col) for col in colunas_numericas if col in cabecalho}

                for linha in linhas_gen:
                    if idx_gestion is not None and linha[idx_gestion] == "LIS":
                        # Converter colunas numéricas
                        for col, idx in idx_numericas.items():
                            try:
                                if linha[idx]:
                                    linha[idx] = str(float(linha[idx])).replace('.', ',')
                            except (ValueError, TypeError):
                                pass
                        writer.writerow(linha)
                        contador += 1
                        total_linhas += 1

                total_convertidos += 1
                print(f"[{numero}/{len(ficheiros)}] {caminho.name} → CSV ({contador:,} linhas)")

            except Exception as erro:
                print(f"[{numero}/{len(ficheiros)}] {caminho.name} — ERRO: {erro}")

    print(f"\n✓ {total_convertidos} ficheiros convertidos")
    print(f"✓ {total_linhas:,} linhas no ficheiro final: {caminho_output}")


[1/1] SAL_DAT027 (21).xls → CSV (8,105 linhas)

✓ 1 ficheiros convertidos
✓ 8,105 linhas no ficheiro final: C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\000.Dados_input\inform_27\SAL_DAT027.csv


In [10]:
# ==========================
# 4. Datas semanais
# ==========================

# Formato FENTREGA: AAAAMMDD
DATA_INICIO_CUSTOS = "20261027"
DATA_FIM_CUSTOS = "20261107"

DATA_INICIO_INGRESSO = "20261101"
DATA_FIM_INGRESSO = "20261107"


In [11]:
# ==========================
# 5. Validar datas
# ==========================

from datetime import datetime


def validar_intervalo(inicio, fim, nome):
    for valor in (inicio, fim):
        if not isinstance(valor, str) or not re.fullmatch(r"\d{8}", valor):
            raise ValueError(f"{nome}: data inválida {valor!r}. Formato: AAAAMMDD.")
        datetime.strptime(valor, "%Y%m%d")
    if inicio > fim:
        raise ValueError(f"{nome}: data inicial posterior à data final.")
    return inicio, fim


intervalo_custos = validar_intervalo(DATA_INICIO_CUSTOS, DATA_FIM_CUSTOS, "Custos")
intervalo_ingresso = validar_intervalo(DATA_INICIO_INGRESSO, DATA_FIM_INGRESSO, "Ingresso")


In [12]:
# ==========================
# 6. Exportar ficheiros semanais
# ==========================

ficheiro_origem = PASTA_FICHEIROS_OUTPUT / "SAL_DAT027.csv"
ficheiro_custos = PASTA_FICHEIROS_OUTPUT / "weekly custos.csv"
ficheiro_ingresso = PASTA_FICHEIROS_OUTPUT / "weekly ingresso.csv"

if not ficheiro_origem.is_file():
    raise FileNotFoundError(f"Ficheiro não encontrado: {ficheiro_origem}")

contadores = {"custos": 0, "ingresso": 0}

with (
    ficheiro_origem.open("r", newline="", encoding="utf-8-sig") as entrada,
    ficheiro_custos.open("w", newline="", encoding="utf-8") as saida_custos,
    ficheiro_ingresso.open("w", newline="", encoding="utf-8") as saida_ingresso,
):
    leitor = csv.DictReader(entrada, delimiter=";")
    if not leitor.fieldnames or "FENTREGA" not in leitor.fieldnames:
        raise ValueError("A coluna FENTREGA não existe no SAL_DAT027.csv.")

    escritor_custos = csv.DictWriter(saida_custos, fieldnames=leitor.fieldnames, delimiter=";")
    escritor_ingresso = csv.DictWriter(saida_ingresso, fieldnames=leitor.fieldnames, delimiter=";")
    escritor_custos.writeheader()
    escritor_ingresso.writeheader()

    for linha in leitor:
        data = (linha.get("FENTREGA") or "").strip()
        if not re.fullmatch(r"\d{8}", data):
            continue

        if intervalo_custos[0] <= data <= intervalo_custos[1]:
            escritor_custos.writerow(linha)
            contadores["custos"] += 1

        if intervalo_ingresso[0] <= data <= intervalo_ingresso[1]:
            escritor_ingresso.writerow(linha)
            contadores["ingresso"] += 1

print(f"✓ Custos: {contadores['custos']:,} linhas → {ficheiro_custos}")
print(f"✓ Ingresso: {contadores['ingresso']:,} linhas → {ficheiro_ingresso}")


✓ Custos: 7,040 linhas → C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\000.Dados_input\inform_27\weekly custos.csv
✓ Ingresso: 3,820 linhas → C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\000.Dados_input\inform_27\weekly ingresso.csv
